# 💳 Credit Card Fraud Detection
### ML Pipeline: EDA → Preprocessing → SMOTE → Model Training → Evaluation
**Author:** Nishita | VIT Bhopal University

**Dataset:** [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Models:** Logistic Regression, Random Forest, XGBoost

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
from imblearn.over_sampling import SMOTE
import joblib, os

sns.set_style('darkgrid')
print('✅ Libraries loaded!')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Class Distribution:')
print(df['Class'].value_counts())
print(f'\nFraud percentage: {df["Class"].mean()*100:.4f}%')
print(f'\nMissing values: {df.isnull().sum().sum()}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class Imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
sns.countplot(x='Class', data=df, palette=['steelblue', 'crimson'], ax=axes[0])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])

# Pie chart
labels = ['Legitimate (99.83%)', 'Fraud (0.17%)']
sizes = df['Class'].value_counts()
axes[1].pie(sizes, labels=labels, colors=['steelblue', 'crimson'],
            autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.show()

In [ ]:
# Transaction Amount by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df['Class']==0]['Amount'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Legitimate Transaction Amounts')
sns.histplot(df[df['Class']==1]['Amount'], bins=50, ax=axes[1], color='crimson')
axes[1].set_title('Fraud Transaction Amounts')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap - top features
corr = df.corr()
top_features = corr['Class'].abs().sort_values(ascending=False).head(15).index
plt.figure(figsize=(12, 8))
sns.heatmap(df[top_features].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap - Top Features')
plt.tight_layout()
plt.show()

## 4. Preprocessing & SMOTE

In [ ]:
scaler = StandardScaler()
df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_Scaled'] = scaler.fit_transform(df[['Time']])
df.drop(['Amount', 'Time'], axis=1, inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
print('Before SMOTE:', pd.Series(y_train).value_counts().to_dict())
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print('After SMOTE:', pd.Series(y_train_sm).value_counts().to_dict())

## 5. Model Training

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob)
    }
    print(f'  Accuracy: {results[name]["accuracy"]:.4f} | ROC-AUC: {results[name]["roc_auc"]:.4f}\n')

## 6. Evaluation & Visualizations

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(f'{name}\nAcc: {res["accuracy"]:.4f}')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.4f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.show()

In [ ]:
# Feature Importance - Random Forest
rf = results['Random Forest']['model']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
top = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, palette='viridis')
plt.title('Top 15 Feature Importances - Random Forest')
plt.xlabel('Importance Score')
plt.show()

## 7. Save Best Model

In [ ]:
best_name = max(results, key=lambda x: results[x]['roc_auc'])
print(f'Best model: {best_name} | ROC-AUC: {results[best_name]["roc_auc"]:.4f}')

os.makedirs('model', exist_ok=True)
joblib.dump(results[best_name]['model'], 'model/best_model.pkl')
joblib.dump(X.columns.tolist(), 'model/feature_columns.pkl')
print('✅ Model saved to model/best_model.pkl')